**import spark**

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.types import *
from pyspark.sql import functions as F

**start session**

In [2]:
spark = SparkSession.builder.appName("E-commerce Pipeline").getOrCreate()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/06/02 12:33:33 WARN Utils: Your hostname, cliffe-HP-Pavilion-Laptop-15-cs3xxx, resolves to a loopback address: 127.0.1.1; using 192.168.1.103 instead (on interface wlo1)
26/06/02 12:33:33 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/06/02 12:33:34 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


**read the data**

In [3]:
customers_df = spark.read.csv("data/customers.csv", header=True)
items_df = spark.read.csv("data/order_items.csv", header=True)
orders_df = spark.read.csv("data/orders.csv", header=True)
returns_df = spark.read.csv("data/returns.csv", header=True)

**view the data for data type transformation**

1. Customers

In [4]:
customers_df.show()

+-----------+-----------+------------+-------------+--------------------+
|customer_id|signup_date|     country|customer_tier|               email|
+-----------+-----------+------------+-------------+--------------------+
|     C00247| 2018-09-10|       Ghana|       Silver|courtneyberger@ex...|
|     C00125| 2023-12-21|    Ethiopia|         Gold|  abrown@example.com|
|     C00413| 2020-12-13|      Rwanda|     platinum|elizabeth18@examp...|
|     C00219| 2018-11-09|      Rwanda|       BRONZE| ugibson@example.org|
|     C00016| 09/05/2021|     Senegal|         Gold|lynchgeorge@examp...|
|     C00093| 06/03/2023|       Kenya|       BRONZE|michael86@example...|
|     C00223| 07/09/2019|South Africa|     platinum|tylerjohnson@exam...|
|     C00458| 2020-11-16|South Africa|       Bronze| sarah52@example.com|
|     C00059| 26/02/2019|      Rwanda|       Silver|  sara74@example.com|
|     C00356| 2019-02-25|     Senegal|       Silver|carlamoore@exampl...|
|     C00417| 2019-06-14|     Senegal|

In [5]:
# Isolate stubborn columns
stubborn_cus = customers_df.filter(
    F.expr("try_to_date(signup_date, 'MM/dd/yyyy')").isNull() & 
    F.expr("try_to_date(signup_date, 'yyyy-MM-dd')").isNull()
)

# Clean rows
customers_df = customers_df.filter(
    F.expr("try_to_date(signup_date, 'MM/dd/yyyy')").isNotNull() | 
    F.expr("try_to_date(signup_date, 'yyyy-MM-dd')").isNotNull()
)

# Cast other columns
customers_df = customers_df.select(
    F.col("customer_id").cast("string"), 
    F.coalesce(
        F.expr("try_to_date(signup_date, 'MM/dd/yyyy')"),
        F.expr("try_to_date(signup_date, 'yyyy-MM-dd')")
    ).alias("signup_date"), 
    F.col("country").cast("string"), 
    F.col("customer_tier").cast("string"), 
    F.col("email").cast("string")
)

In [6]:
# Confirm 
customers_df.show()

+-----------+-----------+------------+-------------+--------------------+
|customer_id|signup_date|     country|customer_tier|               email|
+-----------+-----------+------------+-------------+--------------------+
|     C00247| 2018-09-10|       Ghana|       Silver|courtneyberger@ex...|
|     C00125| 2023-12-21|    Ethiopia|         Gold|  abrown@example.com|
|     C00413| 2020-12-13|      Rwanda|     platinum|elizabeth18@examp...|
|     C00219| 2018-11-09|      Rwanda|       BRONZE| ugibson@example.org|
|     C00016| 2021-09-05|     Senegal|         Gold|lynchgeorge@examp...|
|     C00093| 2023-06-03|       Kenya|       BRONZE|michael86@example...|
|     C00223| 2019-07-09|South Africa|     platinum|tylerjohnson@exam...|
|     C00458| 2020-11-16|South Africa|       Bronze| sarah52@example.com|
|     C00356| 2019-02-25|     Senegal|       Silver|carlamoore@exampl...|
|     C00417| 2019-06-14|     Senegal|         gold|tanner82@example.net|
|     C00222| 2019-04-10|      Uganda|

2. Order Items

In [7]:
items_df.show()

+----------+--------+----------+--------+----------+-----------+
|   item_id|order_id|product_id|quantity|unit_price|   category|
+----------+--------+----------+--------+----------+-----------+
|I000001866|O0000633|     P0289|       6|    364.01|     beauty|
|I000002153|O0000724|     P0460|       5|    395.26|home_garden|
|I000000203|O0000072|     P0249|       6|    327.38|     beauty|
|I000001503|O0000510|     P0341|       7|    496.84|       toys|
|I000002193|O0000743|     P0318|       1|    426.77|     sports|
|I000001284|O0000434|     P0309|       1|     21.39|home_garden|
|I000000560|O0000196|     P0229|       5|    349.46|   clothing|
|I000003401|O0001149|     P0300|       9|    351.22|       toys|
|I000003511|O0001193|     P0225|       2|     39.28|   clothing|
|I000003976|O0001358|     P0089|       9|    326.18|     sports|
|I000005129|O0001736|     P0304|       4|      83.7|      books|
|I000003561|O0001211|     P0376|       6|    467.11|   clothing|
|I000000341|O0000119|    

In [8]:
items_df = items_df.select(
    F.col("item_id").cast("string"), 
    F.col("order_id").cast("string"), 
    F.col("product_id").cast("string"), 
    F.col("quantity").cast("int"), 
    F.col("unit_price").cast("double"), 
    F.col("category").cast("string")
)

In [9]:
items_df.show()

+----------+--------+----------+--------+----------+-----------+
|   item_id|order_id|product_id|quantity|unit_price|   category|
+----------+--------+----------+--------+----------+-----------+
|I000001866|O0000633|     P0289|       6|    364.01|     beauty|
|I000002153|O0000724|     P0460|       5|    395.26|home_garden|
|I000000203|O0000072|     P0249|       6|    327.38|     beauty|
|I000001503|O0000510|     P0341|       7|    496.84|       toys|
|I000002193|O0000743|     P0318|       1|    426.77|     sports|
|I000001284|O0000434|     P0309|       1|     21.39|home_garden|
|I000000560|O0000196|     P0229|       5|    349.46|   clothing|
|I000003401|O0001149|     P0300|       9|    351.22|       toys|
|I000003511|O0001193|     P0225|       2|     39.28|   clothing|
|I000003976|O0001358|     P0089|       9|    326.18|     sports|
|I000005129|O0001736|     P0304|       4|      83.7|      books|
|I000003561|O0001211|     P0376|       6|    467.11|   clothing|
|I000000341|O0000119|    

3. Orders

In [10]:
orders_df.show()

+--------+-----------+----------+---------+------------+------------+
|order_id|customer_id|order_date|   status|total_amount|discount_pct|
+--------+-----------+----------+---------+------------+------------+
|O0001478|     C00173|2023-03-29| refunded|     1239.08|        15.0|
|O0000893|     C00194|02/10/2023|  shipped|      858.91|        25.0|
|O0000231|     C00446|2023-12-04|  pending|      1920.8|        20.0|
|O0000466|     C00342|2024-05-27| refunded|     1432.04|        20.0|
|O0000931|     C00036|23/02/2024|  shipped|         9.3|        NULL|
|O0000249|     C00422|2022-01-28|  shipped|     2390.69|        25.0|
|O0001200|     C00375|30/11/2022| refunded|      881.92|         0.0|
|O0001920|     C00049|2024-06-03|  pending|     1034.52|        20.0|
|O0000937|       NULL|2024-03-14|completed|     1842.99|        10.0|
|O0001488|     C00122|2023-05-11|cancelled|     2094.84|        10.0|
|O0001969|     C00242|2024-04-16|cancelled|      1628.3|        20.0|
|O0000800|     C0042

In [11]:
# Isolate stubborn date rows
stubborn_ord = orders_df.filter(
    F.expr("try_to_date(order_date, 'MM/dd/yyyy')").isNull() &
    F.expr("try_to_date(order_date, 'yyyy-MM-dd')").isNull()
)
# Clean rows
orders_df = orders_df.filter(
    F.expr("try_to_date(order_date, 'MM/dd/yyyy')").isNotNull() |
    F.expr("try_to_date(order_date, 'yyyy-MM-dd')").isNotNull()
)

# Complete dataframe
orders_df = orders_df.select(
    F.col("order_id").cast("string"), 
    F.col("customer_id").cast("string"), 
    F.coalesce(
        F.expr("try_to_date(order_date, 'MM/dd/yyyy')"),
        F.expr("try_to_date(order_date, 'yyyy-MM-dd')")
    ).alias("order_date"),                       
    F.col("status").cast("string"), 
    F.col("total_amount").cast("double"), 
    F.col("discount_pct").cast("double")
)

In [12]:
orders_df.show()

+--------+-----------+----------+---------+------------+------------+
|order_id|customer_id|order_date|   status|total_amount|discount_pct|
+--------+-----------+----------+---------+------------+------------+
|O0001478|     C00173|2023-03-29| refunded|     1239.08|        15.0|
|O0000893|     C00194|2023-02-10|  shipped|      858.91|        25.0|
|O0000231|     C00446|2023-12-04|  pending|      1920.8|        20.0|
|O0000466|     C00342|2024-05-27| refunded|     1432.04|        20.0|
|O0000249|     C00422|2022-01-28|  shipped|     2390.69|        25.0|
|O0001920|     C00049|2024-06-03|  pending|     1034.52|        20.0|
|O0000937|       NULL|2024-03-14|completed|     1842.99|        10.0|
|O0001488|     C00122|2023-05-11|cancelled|     2094.84|        10.0|
|O0001969|     C00242|2024-04-16|cancelled|      1628.3|        20.0|
|O0000800|     C00427|2022-03-11|  pending|     1144.91|        25.0|
|O0000096|     C00187|2022-08-20| refunded|     2463.48|         0.0|
|O0000610|     C0039

4. Returns

In [13]:
returns_df.show()

+-------------+---------------+-----------+----------------+-------------+
|    return_id|       order_id|return_date|          reason|refund_amount|
+-------------+---------------+-----------+----------------+-------------+
|      R000215|       O0000434| 2023-07-25|      wrong_item|      1141.82|
|      R000234|       O0001069| 2024-05-21|       defective|       180.75|
|      R000268|       O0001933| 2024-04-10|    arrived_late|        331.7|
|      R000181|       O0000885| 2022-04-27|not_as_described|       129.94|
|      R000156|       O0001473| 2023-08-11|       defective|       624.34|
|      R000203|       O0001982| 2022-04-15|    arrived_late|      1084.89|
|      R000017|       O0000643| 2023-06-11|       defective|       412.06|
|      R000204|       O0001279| 2022-05-28|not_as_described|       109.94|
|      R000068|       O0001468| 2023-06-20|       defective|      2175.94|
|      R000029|       O0000159| 07/02/2024|    changed_mind|       386.67|
|      R000194|       O00

In [14]:
# Isolate stubbon rows
stubborn_ret = returns_df.filter(
    F.expr("try_to_date(return_date, 'MM/dd/yyyy')").isNull() &
    F.expr("try_to_date(return_date, 'yyyy-MM-dd')").isNull()
)
returns_df = returns_df.filter(
    F.expr("try_to_date(return_date, 'MM/dd/yyyy')").isNotNull() |
    F.expr("try_to_date(return_date, 'yyyy-MM-dd')").isNotNull()
)
returns_df = returns_df.select(
    F.col("return_id").cast("string"), 
    F.col("order_id").cast("string"), 
    F.coalesce(
        F.expr("try_to_date(return_date, 'MM/dd/yyyy')"),
        F.expr("try_to_date(return_date, 'yyyy-MM-dd')")
    ).alias("return_date"),
    F.col("reason").cast("string"), 
    F.col("refund_amount").cast("double"))

In [15]:
returns_df.show()

+-------------+---------------+-----------+----------------+-------------+
|    return_id|       order_id|return_date|          reason|refund_amount|
+-------------+---------------+-----------+----------------+-------------+
|      R000215|       O0000434| 2023-07-25|      wrong_item|      1141.82|
|      R000234|       O0001069| 2024-05-21|       defective|       180.75|
|      R000268|       O0001933| 2024-04-10|    arrived_late|        331.7|
|      R000181|       O0000885| 2022-04-27|not_as_described|       129.94|
|      R000156|       O0001473| 2023-08-11|       defective|       624.34|
|      R000203|       O0001982| 2022-04-15|    arrived_late|      1084.89|
|      R000017|       O0000643| 2023-06-11|       defective|       412.06|
|      R000204|       O0001279| 2022-05-28|not_as_described|       109.94|
|      R000068|       O0001468| 2023-06-20|       defective|      2175.94|
|      R000029|       O0000159| 2024-07-02|    changed_mind|       386.67|
|      R000194|       O00

**Drop duplicates**

In [16]:
# Drop duplicate columns based on customer id
customers_df = customers_df.dropDuplicates(['customer_id'])

In [17]:
# Drop duplicate columns based on item id
items_df = items_df.dropDuplicates(['item_id'])

In [18]:
orders_df = orders_df.dropDuplicates(['order_id'])

**Standardize customer tier to lower case**

In [19]:
customers_df = customers_df.withColumn("customer_tier", F.lower(F.col("customer_tier")))
customers_df.show(5)

[Stage 14:>                                                         (0 + 1) / 1]

+-----------+-----------+-------+-------------+--------------------+
|customer_id|signup_date|country|customer_tier|               email|
+-----------+-----------+-------+-------------+--------------------+
|     C00001| 2018-07-15| Uganda|       bronze|johnsonjoshua@exa...|
|     C00002| 2022-03-08| Uganda|     platinum|jillrhodes@exampl...|
|     C00003| 2023-01-14|  Egypt|     platinum|garzaanthony@exam...|
|     C00004| 2021-11-11|  Kenya|       silver|jesseguzman@examp...|
|     C00005| 2023-02-23| Rwanda|         gold|jennifermiles@exa...|
+-----------+-----------+-------+-------------+--------------------+
only showing top 5 rows


**Drop Null rows**

In [20]:
# Customers
customers_df = customers_df.filter(customers_df["customer_id"].isNotNull())

# Orders
orders_df = orders_df.filter(orders_df["order_id"].isNotNull())

# Order Items
items_df = items_df.filter(items_df["order_id"].isNotNull())

# Returns
returns_df = returns_df.filter(returns_df["order_id"].isNotNull())

**Joins & Enrichment**

In [21]:
customer_orders_df = customers_df.join(
    orders_df,
    on="customer_id",
    how="inner"
)
customer_order_items_df = customer_orders_df.join(
    items_df,
    on="order_id",
    how="inner"
)
anti_items = items_df.join(
    customer_order_items_df,
    on="item_id",
    how="left_anti"
)

customer_order_items_df = customer_order_items_df.withColumn(
    "net_amount",
    F.col("total_amount") * (1 - F.col("discount_pct") / 100))

In [22]:
customer_order_items_df.show()

+--------+-----------+-----------+------------+-------------+--------------------+----------+---------+------------+------------+----------+----------+--------+----------+-----------+------------------+
|order_id|customer_id|signup_date|     country|customer_tier|               email|order_date|   status|total_amount|discount_pct|   item_id|product_id|quantity|unit_price|   category|        net_amount|
+--------+-----------+-----------+------------+-------------+--------------------+----------+---------+------------+------------+----------+----------+--------+----------+-----------+------------------+
|O0000001|     C00224| 2018-10-10|     Nigeria|       silver| molly71@example.net|2023-11-19|  shipped|     2173.29|         0.0|I000000002|     P0212|       6|     73.76|   clothing|           2173.29|
|O0000001|     C00224| 2018-10-10|     Nigeria|       silver| molly71@example.net|2023-11-19|  shipped|     2173.29|         0.0|I000000001|     P0060|       4|    303.12|      books|     

**Aggregations & Window Functions**

In [28]:
from pyspark.sql.window import Window

cust_window = Window.partitionBy("customer_id")

spend_df = customer_order_items_df.withColumn(
    "lifetime_spend",
    F.sum("net_amount").over(cust_window)
)

country_rank = Window.partitionBy("country").orderBy(F.desc("lifetime_spend"))
country_cus_rank = spend_df.withColumn(
    "customer_country_rank", 
    F.dense_rank().over(country_rank)
)
country_cus_rank.show(10)

+--------+-----------+-----------+-------+-------------+--------------------+----------+---------+------------+------------+----------+----------+--------+----------+-----------+------------------+--------------+---------------------+
|order_id|customer_id|signup_date|country|customer_tier|               email|order_date|   status|total_amount|discount_pct|   item_id|product_id|quantity|unit_price|   category|        net_amount|lifetime_spend|customer_country_rank|
+--------+-----------+-----------+-------+-------------+--------------------+----------+---------+------------+------------+----------+----------+--------+----------+-----------+------------------+--------------+---------------------+
|O0000306|     C00132| 2019-02-24|  Egypt|     platinum|meadowsbrittany@e...|2023-07-12|cancelled|     1419.49|        10.0|I000000890|     P0046|       5|    203.87| food_drink|          1277.541|     20707.125|                    1|
|O0000306|     C00132| 2019-02-24|  Egypt|     platinum|mead

In [34]:
# Convert time to seconds
orders_time = customer_order_items_df.withColumn(
    "orders_time_sec",
    F.col("order_date").cast("timestamp").cast("long")
)

# Define the 7-day rolling window
time_frame = 7*24*60*60
rolling_window = Window.partitionBy("customer_id") \
    .orderBy("orders_time_sec") \
    .rangeBetween(-time_frame, Window.currentRow)

rolling_count = orders_time.withColumn(
    "7_day_rolling_orders",
    F.count("order_id").over(rolling_window)
)
rolling_count.show(10)

+--------+-----------+-----------+-------+-------------+--------------------+----------+---------+------------+------------+----------+----------+--------+----------+-----------+------------------+---------------+--------------------+
|order_id|customer_id|signup_date|country|customer_tier|               email|order_date|   status|total_amount|discount_pct|   item_id|product_id|quantity|unit_price|   category|        net_amount|orders_time_sec|7_day_rolling_orders|
+--------+-----------+-----------+-------+-------------+--------------------+----------+---------+------------+------------+----------+----------+--------+----------+-----------+------------------+---------------+--------------------+
|O0001824|     C00001| 2018-07-15| Uganda|       bronze|johnsonjoshua@exa...|2022-08-12|  shipped|       64.65|        25.0|I000005395|     P0285|       4|    195.25| food_drink|48.487500000000004|     1660251600|                   4|
|O0001824|     C00001| 2018-07-15| Uganda|       bronze|john

In [36]:
monthly_df = customer_order_items_df.withColumn(
    "order_month",
    F.trunc("order_date", "month")
)

window_month = Window.partitionBy("order_month")
monthly_df = monthly_df.withColumn(
    "monthly_revenue",
    F.sum("net_amount").over(window_month)
)

category_month = Window.partitionBy("order_month", "category")
monthly_df = monthly_df.withColumn(
    "cat_month_revenue",
    F.sum("net_amount").over(category_month)
)    

category_share = monthly_df.withColumn(
    "cat_revenue_share",
    F.col("cat_month_revenue") / F.col("monthly_revenue")
)

category_share.show()

+--------+-----------+-----------+--------+-------------+--------------------+----------+---------+------------+------------+----------+----------+--------+----------+--------+------------------+-----------+------------------+------------------+--------------------+
|order_id|customer_id|signup_date| country|customer_tier|               email|order_date|   status|total_amount|discount_pct|   item_id|product_id|quantity|unit_price|category|        net_amount|order_month|   monthly_revenue| cat_month_revenue|   cat_revenue_share|
+--------+-----------+-----------+--------+-------------+--------------------+----------+---------+------------+------------+----------+----------+--------+----------+--------+------------------+-----------+------------------+------------------+--------------------+
|O0000672|     C00045| 2018-09-04| Nigeria|       silver|perezrebecca@exam...|2022-01-29|completed|      821.26|         0.0|I000001989|     P0278|       3|    339.48|    NULL|            821.26| 202

**Return Analysis**

In [39]:
order_returns_df = customer_order_items_df.join(
    returns_df,
    on="order_id",
    how="left"
)

rate_df = order_returns_df.groupBy("category", "customer_tier") \
    .agg(F.count("order_id").alias("total_orders"), F.count("return_id").alias("total_returns"))

rate_df = rate_df.withColumn(
    "return_rate",
    F.col("total_returns") / F.col("total_orders")    
)

customer_refunds_df = order_returns_df.groupBy("customer_id") \
    .agg(F.sum("refund_amount").alias("total_refunded")
)

top_10_refund_customers = customer_refunds_df.orderBy(F.desc("total_refunded")).limit(10)

data_df = order_returns_df.withColumn(
    "refund_exceeds_order",
    F.col("refund_amount") > F.col("net_amount")
)

data_df.show()

+--------+-----------+-----------+------------+-------------+--------------------+----------+---------+------------+------------+----------+----------+--------+----------+-----------+------------------+---------+-----------+------------+-------------+--------------------+
|order_id|customer_id|signup_date|     country|customer_tier|               email|order_date|   status|total_amount|discount_pct|   item_id|product_id|quantity|unit_price|   category|        net_amount|return_id|return_date|      reason|refund_amount|refund_exceeds_order|
+--------+-----------+-----------+------------+-------------+--------------------+----------+---------+------------+------------+----------+----------+--------+----------+-----------+------------------+---------+-----------+------------+-------------+--------------------+
|O0000001|     C00224| 2018-10-10|     Nigeria|       silver| molly71@example.net|2023-11-19|  shipped|     2173.29|         0.0|I000000002|     P0212|       6|     73.76|   clothin